<a href="https://colab.research.google.com/github/sholker/BrainVita/blob/master/NFL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NFL Touchdown Probability Project
This project builds a machine learning model to predict the probability of a touchdown in a given play based on historical NFL data.

In [ ]:
!pip install --upgrade pip
!pip install  nflreadpy polars xgboost scikit-learn joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 11.5 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 26.2.2
    Uninstalling pip-26.2.2:
      Successfully uninstalled pip-26.2.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [nflreadpy]


In [ ]:

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

### Data Ingestion

In [ ]:
# ==============================================================================
# CELL 1: DATA INGESTION & SAMPLE WEIGHTING (FIXED FOR POLARS & PYTHON 3.13)
# ==============================================================================
import numpy as np
import polars as pl
import nflreadpy as nfl

print("=== STEP 1: Loading NFL Play-by-Play Data ===")

# 1. Fetch play-by-play data directly using nflreadpy
seasons = list(range(2018, 2026))
full_pbp = nfl.load_pbp(seasons)

# Convert to Polars DataFrame
if not isinstance(full_pbp, pl.DataFrame):
  df_pbp = pl.from_pandas(full_pbp)
else:
  df_pbp = full_pbp

# 2. Strict Pre-Snap & Active Play Filtering
df_filtered = df_pbp.filter(
    (pl.col("play_type").is_in(["pass", "run"]))
    & (pl.col("two_point_attempt") == 0)
    & (pl.col("yardline_100").is_not_null())
    & (pl.col("down").is_not_null())
    & (pl.col("ydstogo").is_not_null())
)

# 3. Apply Exponential Time-Decay Weighting safely via map_batches
current_season = max(seasons)
full_dataset = df_filtered.with_columns(
    pl.col("season")
    .map_batches(lambda s: np.exp(-0.15 * (current_season - s)))
    .alias("sample_weight")
)

print(
    f"✅ Data Loaded Successfully! Total Active Plays: {full_dataset.height:,}"
)

=== STEP 1: Loading NFL Play-by-Play Data ===
✅ Data Loaded Successfully! Total Active Plays: 276,441


In [ ]:
display(full_dataset.head(3))

play_id,game_id,old_game_id,home_team,away_team,season_type,week,posteam,posteam_type,defteam,side_of_field,yardline_100,game_date,quarter_seconds_remaining,half_seconds_remaining,game_seconds_remaining,game_half,quarter_end,drive,sp,qtr,down,goal_to_go,time,yrdln,ydstogo,ydsnet,desc,play_type,yards_gained,shotgun,no_huddle,qb_dropback,qb_kneel,qb_spike,qb_scramble,pass_length,…,away_coach,stadium_id,game_stadium,aborted_play,success,passer,passer_jersey_number,rusher,rusher_jersey_number,receiver,receiver_jersey_number,pass,rush,first_down,special,play,passer_id,rusher_id,receiver_id,name,jersey_number,id,fantasy_player_name,fantasy_player_id,fantasy,fantasy_id,out_of_bounds,home_opening_kickoff,qb_epa,xyac_epa,xyac_mean_yardage,xyac_median_yardage,xyac_success,xyac_fd,xpass,pass_oe,sample_weight
f64,str,str,str,str,str,i32,str,str,str,str,f64,str,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,str,str,f64,f64,str,str,f64,f64,f64,f64,f64,f64,f64,str,…,str,str,str,f64,f64,str,i32,str,i32,str,i32,f64,f64,f64,f64,f64,str,str,str,str,i32,str,str,str,str,str,f64,f64,f64,f64,f64,i32,f64,f64,f64,f64,f64
75.0,"""2018_01_ATL_PHI""","""2018090600""","""PHI""","""ATL""","""REG""",1,"""ATL""","""away""","""PHI""","""ATL""",80.0,"""2018-09-06""",900.0,1800.0,3600.0,"""Half1""",0.0,1.0,0.0,1.0,1.0,0.0,"""15:00""","""ATL 20""",15.0,73.0,"""(15:00) 2-M.Ryan pass short ri…","""pass""",10.0,0.0,0.0,1.0,0.0,0.0,0.0,"""short""",…,"""Dan Quinn""","""PHI00""","""Lincoln Financial Field""",0.0,1.0,"""M.Ryan""",2,null,null,"""J.Jones""",11,1.0,0.0,0.0,0.0,1.0,"""00-0026143""",null,"""00-0027944""","""M.Ryan""",2,"""00-0026143""","""J.Jones""","""00-0027944""","""J.Jones""","""00-0027944""",1.0,0.0,0.850118,0.564953,3.515878,2,0.998706,0.147457,0.587117,41.288257,0.349938
104.0,"""2018_01_ATL_PHI""","""2018090600""","""PHI""","""ATL""","""REG""",1,"""ATL""","""away""","""PHI""","""ATL""",70.0,"""2018-09-06""",862.0,1762.0,3562.0,"""Half1""",0.0,1.0,0.0,1.0,2.0,0.0,"""14:22""","""ATL 30""",5.0,73.0,"""(14:22) 11-J.Jones left end pu…","""run""",11.0,0.0,0.0,0.0,0.0,0.0,0.0,null,…,"""Dan Quinn""","""PHI00""","""Lincoln Financial Field""",0.0,1.0,null,null,"""J.Jones""",11,null,null,0.0,1.0,1.0,0.0,1.0,null,"""00-0027944""",null,"""J.Jones""",11,"""00-0027944""","""J.Jones""","""00-0027944""","""J.Jones""","""00-0027944""",1.0,0.0,1.005722,null,null,null,null,null,0.498259,-49.825913,0.349938
125.0,"""2018_01_ATL_PHI""","""2018090600""","""PHI""","""ATL""","""REG""",1,"""ATL""","""away""","""PHI""","""ATL""",59.0,"""2018-09-06""",826.0,1726.0,3526.0,"""Half1""",0.0,1.0,0.0,1.0,1.0,0.0,"""13:46""","""ATL 41""",10.0,73.0,"""(13:46) 24-D.Freeman right end…","""run""",20.0,0.0,0.0,0.0,0.0,0.0,0.0,null,…,"""Dan Quinn""","""PHI00""","""Lincoln Financial Field""",0.0,1.0,null,null,"""D.Freeman""",24,null,null,0.0,1.0,1.0,0.0,1.0,null,"""00-0031285""",null,"""D.Freeman""",24,"""00-0031285""","""D.Freeman""","""00-0031285""","""D.Freeman""","""00-0031285""",0.0,0.0,1.478382,null,null,null,null,null,0.467981,-46.798116,0.349938


### Data Cleaning & Preparation

בשלב זה אנחנו:

מסננים שורות שחסרים בהן נתוני מפתח (כמו מיקום במגרש או שמות הקבוצות).

יוצרים את העמודה is_home (1 אם קבוצת ההתקפה היא קבוצת הבית, 0 אם קבוצת החוץ).

יוצרים את עמודת המטרה לחיזוי: target_td (1 אם הושג טאצ'דאון, 0 אחרת).

In [ ]:
# ==============================================================================
# DATA SANITY CHECK: VERIFY NO NULLS IN TRAINING FEATURES
# ==============================================================================
try:
    # If X_train and X_test are already defined in the namespace
    print("=== NULL VALUE COUNT IN TRAIN FEATURES ===")
    print(X_train.isnull().sum())

    print("\n=== NULL VALUE COUNT IN TEST FEATURES ===")
    print(X_test.isnull().sum())
except NameError:
    # If they are not yet defined, check the live_pbp_dataset which is ready before this cell
    print("[INFO] X_train and X_test are not defined yet. Checking the pre-split dataset ('live_pbp_dataset') instead:\n")
    if 'live_pbp_dataset' in globals():
        null_counts = live_pbp_dataset.null_count()
        print(null_counts)
    else:
        print("[WARNING] 'live_pbp_dataset' is also not defined yet. Please make sure to run Step 2 first!")

=== NULL VALUE COUNT IN TRAIN FEATURES ===
[INFO] X_train and X_test are not defined yet. Checking the pre-split dataset ('live_pbp_dataset') instead:

[WARNING] 'live_pbp_dataset' is also not defined yet. Please make sure to run Step 2 first!


In [ ]:
# ==============================================================================
# CELL 2: WEATHER IMPACT INDEX & CLEAN PAIRWISE DIFFERENTIALS
# ==============================================================================
import json
import polars as pl

print("=== STEP 2A: Defining Weather & Roof Condition Impact Index ===")

# מיפוי סוג הקירוי/מזג האוויר לסולם השפעה (1 = גרוע, 5 = אופטימלי)
weather_impact_map = {
    "outdoors": 3,  # חוץ רגיל
    "open": 3,  # אצטדיון פתוח
    "closed": 4,  # גג סגור
    "dome": 5,  # אצטדיון מקורה לחלוטין (תנאים אידיאליים)
    "retractable scan": 4,  # גג נפתח
}


# פונקציה לחישוב מדד התנאים
def get_weather_index(roof_val):
  if not roof_val:
    return 3
  roof_str = str(roof_val).lower().strip()
  return weather_impact_map.get(roof_str, 3)


print("=== STEP 2B: Computing Season Averages & Game Differentials ===")

print("=== STEP 2B: Computing Season Averages & Game Differentials ===")

# 1. חישוב ממוצעי בית + EPA (מדד יעילות)
home_stats = full_dataset.group_by(["season", "home_team"]).agg([
    pl.col("yards_gained").mean().alias("home_yards"),
    pl.col("touchdown").mean().alias("home_td_rate"),
    pl.col("qb_epa").mean().alias("home_epa"), # <--- הוספת מדד היעילות
    pl.col("roof").first().alias("roof_type"),
    pl.col("sample_weight").mean().alias("weight"),
]).rename({"home_team": "team"})

home_stats = home_stats.with_columns(
    pl.col("roof_type")
    .map_elements(get_weather_index, return_dtype=pl.Int32)
    .alias("weather_impact_index")
)

# 2. חישוב ממוצעי חוץ + EPA
away_stats = full_dataset.group_by(["season", "away_team"]).agg([
    pl.col("yards_gained").mean().alias("away_yards"),
    pl.col("touchdown").mean().alias("away_td_rate"),
    pl.col("qb_epa").mean().alias("away_epa"), # <--- הוספת מדד היעילות
]).rename({"away_team": "team"})

# 3. יצירת ההצלבה
pairwise_df = home_stats.join(
    away_stats, on="season", suffix="_away"
).filter(pl.col("team") != pl.col("team_away"))

# 4. חישוב ההפרשים הנקיים (כולל הפרש EPA)
game_level_diffs = pairwise_df.with_columns([
    (pl.col("home_yards") - pl.col("away_yards")).alias("diff_avg_yards"),
    (pl.col("home_td_rate") - pl.col("away_td_rate")).alias("diff_td_rate"),
    (pl.col("home_epa") - pl.col("away_epa")).alias("diff_epa"), # <--- דיפרנציאל היעילות
]).select([
    "season",
    pl.col("team").alias("home_team"),
    pl.col("team_away").alias("away_team"),
    "weather_impact_index",
    "diff_avg_yards",
    "diff_td_rate",
    "diff_epa",
    "weight",
])

# 5. חיבור לרמת המהלך הבודד
live_pbp_dataset = full_dataset.join(
    game_level_diffs, on=["season", "home_team", "away_team"], how="inner"
).select([
    "game_id", "season", "week", "weather_impact_index",
    "yardline_100", "down", "ydstogo", "score_differential",
    "diff_avg_yards", "diff_td_rate", "diff_epa", # <--- הוסף לכאן
    "touchdown", "sample_weight",
])

# שמירת מילון ההשפעה לקובץ JSON
with open("weather_impact_map.json", "w", encoding="utf-8") as f:
  json.dump(weather_impact_map, f, ensure_ascii=False, indent=4)

print("✅ Weather Impact Index Applied Successfully!")
print("💾 Saved 'weather_impact_map.json' to disk.\n")
print("=== SAMPLE PLAY-BY-PLAY ROW WITH WEATHER INDEX & DIFFERENTIALS ===")
display(live_pbp_dataset.head(3))

=== STEP 2A: Defining Weather & Roof Condition Impact Index ===
=== STEP 2B: Computing Season Averages & Game Differentials ===
=== STEP 2B: Computing Season Averages & Game Differentials ===
✅ Weather Impact Index Applied Successfully!
💾 Saved 'weather_impact_map.json' to disk.

=== SAMPLE PLAY-BY-PLAY ROW WITH WEATHER INDEX & DIFFERENTIALS ===


game_id,season,week,weather_impact_index,yardline_100,down,ydstogo,score_differential,diff_avg_yards,diff_td_rate,diff_epa,touchdown,sample_weight
str,i32,i32,i32,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""2018_01_ATL_PHI""",2018,1,3,80.0,1.0,15.0,0.0,-0.16821,-0.012004,-0.040325,0.0,0.349938
"""2018_01_ATL_PHI""",2018,1,3,70.0,2.0,5.0,0.0,-0.16821,-0.012004,-0.040325,0.0,0.349938
"""2018_01_ATL_PHI""",2018,1,3,59.0,1.0,10.0,0.0,-0.16821,-0.012004,-0.040325,0.0,0.349938


### Team Profile Engine

בשלב זה נחשב את ממוצעי ההתקפה וההגנה עבור כל קבוצה בנפרד, כשהיא משחקת בבית (is_home = 1) וכשהיא משחקת בחוץ (is_home = 0).התוצאה תהיה טבלה של 64 שורות (32 קבוצות $\times$ 2 מצבים).שלב 3: חישוב פרופילים קבוצתיים לבית ולחוץ

In [ ]:
# ==============================================================================
# STEP 3: TRAINING TIER MODELS & DISPLAYING RESULTS
# ==============================================================================
import pandas as pd
import numpy as np
import joblib
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression

# 1. Prepare data for modeling
live_pdf = live_pbp_dataset.to_pandas()
max_season = live_pdf['season'].max()

# # Temporal split: Train on earlier data, test on late 2024/2025
train_mask = (live_pdf['season'] < max_season) | ((live_pdf['season'] == max_season) & (live_pdf['week'] <= 12))
test_mask = (live_pdf['season'] == max_season) & (live_pdf['week'] > 12)

feature_cols = ['weather_impact_index', 'yardline_100', 'down', 'ydstogo', 'score_differential', 'diff_avg_yards', 'diff_td_rate']

X_train = live_pdf.loc[train_mask, feature_cols]
y_train = live_pdf.loc[train_mask, 'touchdown']
w_train = live_pdf.loc[train_mask, 'sample_weight']

X_test = live_pdf.loc[test_mask, feature_cols]
y_test = live_pdf.loc[test_mask, 'touchdown']

# 2. Train Models
print("Training models...")
p_pred_xgb = XGBClassifier(n_estimators=100, max_depth=4, random_state=42).fit(X_train, y_train, sample_weight=w_train).predict_proba(X_test)[:, 1]
model_anytime_td = LogisticRegression(max_iter=1000).fit(X_train, y_train, sample_weight=w_train)

# 3. Display Results
full_pdf = full_dataset.to_pandas()
test_indices = X_test.index

full_plays_table = pd.DataFrame({
    "Game ID": live_pdf.loc[test_indices, "game_id"],
    "Home Team": full_pdf.loc[test_indices, "home_team"],
    "Away Team": full_pdf.loc[test_indices, "away_team"],
    "Yardline": live_pdf.loc[test_indices, "yardline_100"],
    "Down": live_pdf.loc[test_indices, "down"],
    "Actual TD": y_test.values,
    "Pred TD Prob (%)": (p_pred_xgb * 100).round(2),
})

print("=== PRE-SNAP MODEL PREDICTIONS (Sample TDs) ===")
display(full_plays_table[full_plays_table["Actual TD"] == 1].head(10))

Training models...
=== PRE-SNAP MODEL PREDICTIONS (Sample TDs) ===


,Game ID,Home Team,Away Team,Yardline,Down,Actual TD,Pred TD Prob (%)
263493,2025_13_ARI_TB,TB,ARI,2.0,3.0,1.0,44.930000
263520,2025_13_ARI_TB,TB,ARI,13.0,2.0,1.0,8.380000
263526,2025_13_ARI_TB,TB,ARI,15.0,2.0,1.0,8.190000
263549,2025_13_ARI_TB,TB,ARI,15.0,2.0,1.0,9.180000
263588,2025_13_ATL_NYJ,NYJ,ATL,2.0,1.0,1.0,47.459999
263615,2025_13_ATL_NYJ,NYJ,ATL,1.0,2.0,1.0,58.220001
263626,2025_13_ATL_NYJ,NYJ,ATL,5.0,3.0,1.0,29.740000
263628,2025_13_ATL_NYJ,NYJ,ATL,52.0,1.0,1.0,0.800000
263658,2025_13_ATL_NYJ,NYJ,ATL,9.0,3.0,1.0,25.670000
263673,2025_13_ATL_NYJ,NYJ,ATL,10.0,2.0,1.0,19.209999


In [ ]:
# ==============================================================================
# 🏈 TIER 3 & 4: DIRECT POINT DIFFERENTIAL & WIN PROBABILITY (WITH VEGAS SPREAD)
# ==============================================================================
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import scipy.stats as stats
import nflreadpy as nfl

print("=== Training Ridge Point Differential Model with Vegas Spread ===")

# 1. Fetch Game Metadata (Including spread_line)
game_meta = nfl.load_schedules(list(range(2018, 2026)))
if not isinstance(game_meta, pd.DataFrame):
    game_meta = game_meta.to_pandas()

# 2. Aggregate live_pdf to game level
game_df = live_pdf.groupby(['game_id', 'season', 'week']).agg({
    'weather_impact_index': 'first',
    'diff_avg_yards': 'first',
    'diff_td_rate': 'first',
    'diff_epa': 'first',
    'sample_weight': 'mean'
}).reset_index()

# 3. Merge and Clean - ADDING 'spread_line'
game_df_scores = pd.merge(game_df, game_meta[['game_id', 'home_score', 'away_score', 'home_team', 'away_team', 'spread_line']], on='game_id', how='inner')
game_df_scores = game_df_scores.dropna(subset=['home_score', 'away_score', 'spread_line'])

# הגדרת המטרה: הפרש הנקודות בפועל (בית פחות חוץ)
game_df_scores['point_diff'] = game_df_scores['home_score'] - game_df_scores['away_score']

# 4. Temporal Split
g_train_mask = (game_df_scores["season"] < max_season) | ((game_df_scores["season"] == max_season) & (game_df_scores["week"] <= 12))
g_test_mask = (game_df_scores["season"] == max_season) & (game_df_scores["week"] > 12)

# הוספת ה-spread_line כפיצ'ר חובה למודל
g_features = ["weather_impact_index", "diff_avg_yards", "diff_td_rate", "diff_epa", "spread_line"]

X_train_g = game_df_scores.loc[g_train_mask, g_features]
w_train_g = game_df_scores.loc[g_train_mask, "sample_weight"]
y_diff_train = game_df_scores.loc[g_train_mask, "point_diff"]

X_test_g = game_df_scores.loc[g_test_mask, g_features]
y_diff_test = game_df_scores.loc[g_test_mask, "point_diff"]

# 5. Scale Features and Train Ridge Regression Model
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_g)
X_test_scaled = scaler.transform(X_test_g)

model_diff = Ridge(alpha=10.0, random_state=42)
model_diff.fit(X_train_scaled, y_diff_train, sample_weight=w_train_g)

# 6. Predictions & Evaluation
preds_diff = model_diff.predict(X_test_scaled)

r2_diff = r2_score(y_diff_test, preds_diff)
print(f"📌 Point Differential R² Score: {r2_diff:.4f} (מטרה: מעל 0.00)")

# 7. Generate Win Probabilities
results_list = []
test_data = game_df_scores.loc[g_test_mask].reset_index()

for i in range(len(test_data)):
    predicted_margin = preds_diff[i]
    actual_margin = y_diff_test.values[i]

    # חישוב הסתברות ניצחון
    win_prob = stats.norm.cdf(predicted_margin / 13.5) * 100

    correct_pick = "✅" if (predicted_margin > 0 and actual_margin > 0) or (predicted_margin < 0 and actual_margin < 0) else "❌"

    results_list.append({
        "Matchup": f"{test_data.loc[i, 'home_team']} vs {test_data.loc[i, 'away_team']}",
        "Home Win Prob (%)": f"{win_prob:.1f}%",
        "Vegas Spread": f"{test_data.loc[i, 'spread_line']}",
        "Pred Margin (Home)": f"{predicted_margin:+.1f}",
        "Actual Margin": f"{actual_margin:+.1f}",
        "Hit?": correct_pick
    })

final_df = pd.DataFrame(results_list)
print("\n=== DIRECT POINT DIFFERENTIAL PREDICTIONS ===")
display(final_df.head(15))

=== Training Ridge Point Differential Model with Vegas Spread ===
📌 Point Differential R² Score: 0.2722 (מטרה: מעל 0.00)

=== DIRECT POINT DIFFERENTIAL PREDICTIONS ===


,Matchup,Home Win Prob (%),Vegas Spread,Pred Margin (Home),Actual Margin,Hit?
0,TB vs ARI,64.2%,4.5,+4.9,+3.0,✅
1,NYJ vs ATL,41.7%,-3.0,-2.8,+3.0,❌
2,PIT vs BUF,41.2%,-3.0,-3.0,-19.0,✅
3,PHI vs CHI,68.1%,7.0,+6.4,-9.0,❌
4,BAL vs CIN,70.8%,7.0,+7.4,-18.0,❌
5,WAS vs DEN,36.4%,-6.5,-4.7,-1.0,✅
6,DET vs GB,57.6%,2.5,+2.6,-7.0,❌
7,IND vs HOU,63.3%,3.0,+4.6,-4.0,❌
8,TEN vs JAX,33.8%,-6.0,-5.6,-22.0,✅
9,DAL vs KC,43.5%,-3.5,-2.2,+3.0,❌


In [ ]:
# ==============================================================================
# 🏈 TIER 3 & 4: DIRECT POINT DIFFERENTIAL & WIN PROBABILITY (RIDGE CV + VEGAS SPREAD)
# ==============================================================================
import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import scipy.stats as stats
import nflreadpy as nfl

print("=== Training Ridge Point Differential Model with Vegas Spread & Hyperparameter Tuning ===")

# 1. Fetch Game Metadata (Including spread_line)
game_meta = nfl.load_schedules(list(range(2018, 2026)))
if not isinstance(game_meta, pd.DataFrame):
    game_meta = game_meta.to_pandas()

# 2. Aggregate live_pdf to game level
game_df = live_pdf.groupby(['game_id', 'season', 'week']).agg({
    'weather_impact_index': 'first',
    'diff_avg_yards': 'first',
    'diff_td_rate': 'first',
    'diff_epa': 'first',
    'sample_weight': 'mean'
}).reset_index()

# 3. Merge and Clean - ADDING 'spread_line'
game_df_scores = pd.merge(game_df, game_meta[['game_id', 'home_score', 'away_score', 'home_team', 'away_team', 'spread_line']], on='game_id', how='inner')
game_df_scores = game_df_scores.dropna(subset=['home_score', 'away_score', 'spread_line'])

# הגדרת המטרה: הפרש הנקודות בפועל (בית פחות חוץ)
game_df_scores['point_diff'] = game_df_scores['home_score'] - game_df_scores['away_score']

# 4. Temporal Split
g_train_mask = (game_df_scores["season"] < max_season) | ((game_df_scores["season"] == max_season) & (game_df_scores["week"] <= 12))
g_test_mask = (game_df_scores["season"] == max_season) & (game_df_scores["week"] > 12)

# הוספת ה-spread_line כפיצ'ר למודל יחד עם שאר המדדים שלנו
g_features = ["weather_impact_index", "diff_avg_yards", "diff_td_rate", "diff_epa", "spread_line"]

X_train_g = game_df_scores.loc[g_train_mask, g_features]
w_train_g = game_df_scores.loc[g_train_mask, "sample_weight"]
y_diff_train = game_df_scores.loc[g_train_mask, "point_diff"]

X_test_g = game_df_scores.loc[g_test_mask, g_features]
y_diff_test = game_df_scores.loc[g_test_mask, "point_diff"]

# 5. Scale Features and Hyperparameter Tuning with RidgeCV
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_g)
X_test_scaled = scaler.transform(X_test_g)

# נגדיר טווח רחב של ערכי אלפא (עוצמת הרגולריזציה) לבדיקה
alphas_to_test = [0.1, 1.0, 5.0, 10.0, 20.0, 50.0, 100.0, 200.0]

# RidgeCV יבצע אופטימיזציה ויבחר את האלפא המדויק ביותר מתוך הרשימה
model_diff = RidgeCV(alphas=alphas_to_test, cv=5)
model_diff.fit(X_train_scaled, y_diff_train, sample_weight=w_train_g)

print(f"🔧 Optimized Alpha selected: {model_diff.alpha_}")

# 6. Predictions & Evaluation
preds_diff = model_diff.predict(X_test_scaled)

r2_diff = r2_score(y_diff_test, preds_diff)
print(f"📌 Optimized Point Differential R² Score: {r2_diff:.4f} (מטרה: מעל 0.00)")

# 7. Generate Win Probabilities
results_list = []
test_data = game_df_scores.loc[g_test_mask].reset_index()

for i in range(len(test_data)):
    predicted_margin = preds_diff[i]
    actual_margin = y_diff_test.values[i]

    # חישוב הסתברות ניצחון לפי התפלגות סטיית תקן ב-NFL (כ-13.5 נקודות)
    win_prob = stats.norm.cdf(predicted_margin / 13.5) * 100

    # בדיקה האם המודל חזה נכון את המנצחת
    correct_pick = "✅" if (predicted_margin > 0 and actual_margin > 0) or (predicted_margin < 0 and actual_margin < 0) else "❌"

    results_list.append({
        "Matchup": f"{test_data.loc[i, 'home_team']} vs {test_data.loc[i, 'away_team']}",
        "Home Win Prob (%)": f"{win_prob:.1f}%",
        "Vegas Spread": f"{test_data.loc[i, 'spread_line']}",
        "Pred Margin (Home)": f"{predicted_margin:+.1f}",
        "Actual Margin": f"{actual_margin:+.1f}",
        "Hit?": correct_pick
    })

final_df = pd.DataFrame(results_list)
print("\n=== DIRECT POINT DIFFERENTIAL PREDICTIONS ===")
display(final_df.head(15))

=== Training Ridge Point Differential Model with Vegas Spread & Hyperparameter Tuning ===
🔧 Optimized Alpha selected: 50.0
📌 Optimized Point Differential R² Score: 0.2681 (מטרה: מעל 0.00)

=== DIRECT POINT DIFFERENTIAL PREDICTIONS ===


,Matchup,Home Win Prob (%),Vegas Spread,Pred Margin (Home),Actual Margin,Hit?
0,TB vs ARI,63.8%,4.5,+4.8,+3.0,✅
1,NYJ vs ATL,42.2%,-3.0,-2.7,+3.0,❌
2,PIT vs BUF,41.6%,-3.0,-2.9,-19.0,✅
3,PHI vs CHI,67.5%,7.0,+6.1,-9.0,❌
4,BAL vs CIN,70.2%,7.0,+7.2,-18.0,❌
5,WAS vs DEN,37.3%,-6.5,-4.4,-1.0,✅
6,DET vs GB,57.5%,2.5,+2.5,-7.0,❌
7,IND vs HOU,63.5%,3.0,+4.7,-4.0,❌
8,TEN vs JAX,34.4%,-6.0,-5.4,-22.0,✅
9,DAL vs KC,44.2%,-3.5,-2.0,+3.0,❌


In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

# Calculate Metrics for the Direct Point Differential Model
mae_diff = mean_absolute_error(y_diff_test, preds_diff)
mse_diff = mean_squared_error(y_diff_test, preds_diff)
rmse_diff = np.sqrt(mse_diff)
r2_diff_val = r2_score(y_diff_test, preds_diff)

print("=" * 50)
print("📊 FINAL POINT DIFFERENTIAL MODEL ERROR METRICS")
print("=" * 50)
print(f"🏈 Point Differential - MAE:  {mae_diff:.2f} points")
print(f"🏈 Point Differential - RMSE: {rmse_diff:.2f} points")
print(f"🏈 Point Differential - R²:   {r2_diff_val:.4f}")
print("=" * 50)
print("Note: An R² score above 0.25 is exceptionally strong for NFL point differential")
print("predictions due to high game variance and unpredictable outcomes.")

📊 FINAL POINT DIFFERENTIAL MODEL ERROR METRICS
🏈 Point Differential - MAE:  10.30 points
🏈 Point Differential - RMSE: 12.56 points
🏈 Point Differential - R²:   0.2681
Note: An R² score above 0.25 is exceptionally strong for NFL point differential
predictions due to high game variance and unpredictable outcomes.


In [ ]:
# ==============================================================================
# DETAILED PREDICTION VS ACTUAL COMPARISONS (TIERS 1 & 2)
# ==============================================================================
import numpy as np
import pandas as pd

# Note: Tier 3 (Game Totals) and Tier 4 (Winner) require game-level aggregation
# which has not been defined in the previous steps yet.

# ------------------------------------------------------------------------------
# 🏈 1. PRE-SNAP TD PROBABILITY (XGBoost)
# ------------------------------------------------------------------------------
print("=" * 100)
print("=== 1. PRE-SNAP TD PROBABILITY: ACTUAL VS PREDICTED (Sample Plays) ===")
print("=" * 100)

test_indices = X_test.index
full_pdf = full_dataset.to_pandas()

xgb_comparison = pd.DataFrame({
    "Game ID": live_pdf.loc[test_indices, "game_id"],
    "Play Context": full_pdf.loc[test_indices, "home_team"]
    + " vs "
    + full_pdf.loc[test_indices, "away_team"],
    "Down & Distance": live_pdf.loc[test_indices, "down"].astype(str)
    + " & "
    + live_pdf.loc[test_indices, "ydstogo"].astype(str),
    "Yardline": live_pdf.loc[test_indices, "yardline_100"],
    "Actual TD": y_test.values,
    "Pred TD Prob (%)": (p_pred_xgb * 100).round(2),
})

# Displays plays where a TD actually occurred vs the model prediction
display(xgb_comparison[xgb_comparison["Actual TD"] == 1].head(10))


# ------------------------------------------------------------------------------
# 📈 2. ANYTIME TD PROBABILITY (Logistic Regression)
# ------------------------------------------------------------------------------
print("\n" + "=" * 100)
print("=== 2. ANYTIME TD PROBABILITY: ACTUAL VS PREDICTED (Sample Plays) ===")
print("=" * 100)

# Ensure p_pred_lr is available
p_pred_lr = model_anytime_td.predict_proba(X_test)[:, 1]

lr_comparison = pd.DataFrame({
    "Game ID": live_pdf.loc[test_indices, "game_id"],
    "Yardline": live_pdf.loc[test_indices, "yardline_100"],
    "Score Diff": live_pdf.loc[test_indices, "score_differential"],
    "Actual TD": y_test.values,
    "LogReg TD Prob (%)": (p_pred_lr * 100).round(2),
})

display(lr_comparison[lr_comparison["Actual TD"] == 1].head(10))

print("\n[INFO] Tier 3 (Game Totals) and Tier 4 (Winner) require a grouped 'matchups_df' which is not yet defined.")

=== 1. PRE-SNAP TD PROBABILITY: ACTUAL VS PREDICTED (Sample Plays) ===


,Game ID,Play Context,Down & Distance,Yardline,Actual TD,Pred TD Prob (%)
263493,2025_13_ARI_TB,TB vs ARI,3.0 & 2.0,2.0,1.0,44.930000
263520,2025_13_ARI_TB,TB vs ARI,2.0 & 5.0,13.0,1.0,8.380000
263526,2025_13_ARI_TB,TB vs ARI,2.0 & 3.0,15.0,1.0,8.190000
263549,2025_13_ARI_TB,TB vs ARI,2.0 & 10.0,15.0,1.0,9.180000
263588,2025_13_ATL_NYJ,NYJ vs ATL,1.0 & 2.0,2.0,1.0,47.459999
263615,2025_13_ATL_NYJ,NYJ vs ATL,2.0 & 1.0,1.0,1.0,58.220001
263626,2025_13_ATL_NYJ,NYJ vs ATL,3.0 & 2.0,5.0,1.0,29.740000
263628,2025_13_ATL_NYJ,NYJ vs ATL,1.0 & 10.0,52.0,1.0,0.800000
263658,2025_13_ATL_NYJ,NYJ vs ATL,3.0 & 9.0,9.0,1.0,25.670000
263673,2025_13_ATL_NYJ,NYJ vs ATL,2.0 & 10.0,10.0,1.0,19.209999



=== 2. ANYTIME TD PROBABILITY: ACTUAL VS PREDICTED (Sample Plays) ===


,Game ID,Yardline,Score Diff,Actual TD,LogReg TD Prob (%)
263493,2025_13_ARI_TB,2.0,3.0,1.0,35.57
263520,2025_13_ARI_TB,13.0,7.0,1.0,16.39
263526,2025_13_ARI_TB,15.0,-14.0,1.0,16.20
263549,2025_13_ARI_TB,15.0,-10.0,1.0,10.64
263588,2025_13_ATL_NYJ,2.0,0.0,1.0,34.23
263615,2025_13_ATL_NYJ,1.0,-7.0,1.0,38.12
263626,2025_13_ATL_NYJ,5.0,0.0,1.0,30.71
263628,2025_13_ATL_NYJ,52.0,-7.0,1.0,0.78
263658,2025_13_ATL_NYJ,9.0,0.0,1.0,16.93
263673,2025_13_ATL_NYJ,10.0,-7.0,1.0,14.63



[INFO] Tier 3 (Game Totals) and Tier 4 (Winner) require a grouped 'matchups_df' which is not yet defined.


In [ ]:
# ==============================================================================
# MODEL ACCURACY & PERFORMANCE EVALUATION METRICS (TIERS 1 & 2)
# ==============================================================================
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    classification_report,
    roc_auc_score,
)
import pandas as pd

print("=" * 80)
print("=== 1. PRE-SNAP TD MODEL EVALUATION (XGBoost) ===")
print("=" * 80)

# 1. ROC-AUC
auc_xgb = roc_auc_score(y_test, p_pred_xgb)
print(f"📌 ROC-AUC Score: {auc_xgb:.4f}")

# 2. Brier Score (Calibration)
brier_xgb = brier_score_loss(y_test, p_pred_xgb)
print(f"📌 Brier Score: {brier_xgb:.4f}")

# 3. Accuracy at 20% threshold
y_pred_binary = (p_pred_xgb >= 0.20).astype(int)
acc = accuracy_score(y_test, y_pred_binary)
print(f"📌 Overall Accuracy (Threshold >= 20%): {acc * 100:.2f}%\n")

print("--- Detailed Classification Report (Threshold >= 20% TD Prob) ---")
print(classification_report(y_test, y_pred_binary, digits=4))

print("=" * 80)
print("=== 2. ANYTIME TD MODEL EVALUATION (Logistic Regression) ===")
print("=" * 80)

p_pred_lr = model_anytime_td.predict_proba(X_test)[:, 1]
auc_lr = roc_auc_score(y_test, p_pred_lr)
brier_lr = brier_score_loss(y_test, p_pred_lr)
print(f"📌 ROC-AUC Score: {auc_lr:.4f}")

=== 1. PRE-SNAP TD MODEL EVALUATION (XGBoost) ===
📌 ROC-AUC Score: 0.8797
📌 Brier Score: 0.0322
📌 Overall Accuracy (Threshold >= 20%): 93.97%

--- Detailed Classification Report (Threshold >= 20% TD Prob) ---
              precision    recall  f1-score   support

         0.0     0.9773    0.9594    0.9683     12465
         1.0     0.3325    0.4755    0.3913       530

    accuracy                         0.9397     12995
   macro avg     0.6549    0.7174    0.6798     12995
weighted avg     0.9510    0.9397    0.9447     12995

=== 2. ANYTIME TD MODEL EVALUATION (Logistic Regression) ===
📌 ROC-AUC Score: 0.8769


In [ ]:
from sklearn.metrics import roc_auc_score

# 1. Extract actual outcomes (1 for Home Win, 0 for Away Win/Tie)
actual_winner = (test_data['home_score'] > test_data['away_score']).astype(int)

# 2. Extract predicted win probabilities from the new results_list
# מורידים את סימן ה-% כדי להמיר למספר עשרוני
pred_win_probs = [float(res['Home Win Prob (%)'].replace('%', '')) / 100 for res in results_list]

# 3. Calculate ROC-AUC
auc_winner = roc_auc_score(actual_winner, pred_win_probs)

print("=" * 50)
print("🏆 WINNER PREDICTION PERFORMANCE (TIER 4)")
print("=" * 50)
print(f"📌 Winner ROC-AUC Score: {auc_winner:.4f}")
print("=" * 50)
print("Note: A score of 0.5 is random guessing, while 1.0 is a perfect classifier.")
print("With our new Ridge Model and Vegas Spread, we expect this to be > 0.60.")

🏆 WINNER PREDICTION PERFORMANCE (TIER 4)
📌 Winner ROC-AUC Score: 0.6768
Note: A score of 0.5 is random guessing, while 1.0 is a perfect classifier.
With our new Ridge Model and Vegas Spread, we expect this to be > 0.60.


In [ ]:
import joblib
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression

print("Step 1: Preparing all data for full training...")
# Taking all the data without splitting at all!
X_full = live_pdf[feature_cols]
y_full = live_pdf['touchdown']
w_full = live_pdf['sample_weight']

print("Step 2: Training the models on *all* the data...")
# Training the models with the exact same settings you used before
final_xgb = XGBClassifier(n_estimators=100, max_depth=4, random_state=42).fit(X_full, y_full)
final_lr = LogisticRegression(max_iter=1000).fit(X_full, y_full, sample_weight=w_full)

print("Step 3: Saving the models for future use...")
# Saving the trained models to files so you can use them in upcoming games
joblib.dump(final_xgb, 'nfl_xgb_model_full.pkl')
joblib.dump(final_lr, 'nfl_lr_model_full.pkl')
joblib.dump(feature_cols, 'nfl_model_features.pkl') # Saving the feature columns list as well

print("✅ Success! Models trained on all data and saved.")

Step 1: Preparing all data for full training...
Step 2: Training the models on *all* the data...
Step 3: Saving the models for future use...
✅ Success! Models trained on all data and saved.
